# CSC 4792: Data Mining and Warehousing — Mini Project
## Team #41 — Monze Town Council Dataset

**Council:** Monze Town Council
**Source:** https://www.monzecouncil.gov.zm
**Team:** Project Team #41

This notebook documents the full pipeline used to build the dataset submitted
to Kaggle: web data acquisition, cleaning/preprocessing, and curation of a
structured CDF (Constituency Development Fund) projects table.

**Pipeline overview:**
1. Acquire raw data from the council's website via its WordPress REST API
2. Clean and flatten the raw JSON into readable CSV tables
3. Scan the cleaned text to locate financial/project figures
4. Curate a structured CDF projects dataset from those figures
5. Validate the final output files against the assignment's formatting rules


## 1. Data Acquisition

### 1.1 Source investigation

Monze Town Council's website (`monzecouncil.gov.zm`) is built on WordPress.
Two things were discovered during investigation before writing any scraping code:

- The site does **not** support "pretty permalinks" — the standard REST API
  path `/wp-json/wp/v2/posts` returns a 404. The working route instead uses
  WordPress's query-string fallback: `/?rest_route=/wp/v2/posts`.
- The site's SSL certificate is currently expired
  (`NET::ERR_CERT_DATE_INVALID`), which is common for some Zambian government
  sites. This does not affect the legitimacy of the site or its data — it
  only means certificate verification has to be explicitly disabled when
  connecting to this specific, manually-verified domain.

Using the REST API instead of parsing rendered HTML gives clean, structured
JSON directly (title, publish date, full content, permalink) with no need
for fragile HTML-parsing logic to locate article text on the page.

### 1.2 Scraper

The scraper below pages through the `posts` and `pages` REST endpoints and
saves the complete raw JSON response for each to disk. Keeping the raw JSON
as an intermediate artifact (rather than cleaning inline) makes the pipeline
reproducible and auditable — if the cleaning logic changes later, it can be
re-run against the same raw snapshot without hitting the website again.


In [ ]:
import requests
import json
import time
import os
import urllib3

# Disable SSL verification warnings — see note above on the site's expired cert
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.monzecouncil.gov.zm/?rest_route=/wp/v2"
RAW_DIR = "raw_data"
HEADERS = {
    "User-Agent": "UNZA-CSC4792-Team41-ResearchBot/1.0 (academic mini project)"
}
DELAY_SECONDS = 1.0  # polite delay between paginated requests


def fetch_all(endpoint: str, per_page: int = 100):
    """Fetch every item from a paginated WordPress REST API endpoint."""
    all_items = []
    page = 1
    while True:
        url = f"{BASE_URL}/{endpoint}"
        params = {"per_page": per_page, "page": page}
        resp = requests.get(url, headers=HEADERS, params=params, timeout=20, verify=False)

        if resp.status_code == 400:
            break  # WP returns 400 once you page past the last page
        resp.raise_for_status()

        items = resp.json()
        if not items:
            break

        all_items.extend(items)
        print(f"  {endpoint}: fetched page {page} ({len(items)} items, {len(all_items)} total)")

        total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
        if page >= total_pages:
            break
        page += 1
        time.sleep(DELAY_SECONDS)

    return all_items


def save_json(data, filename):
    os.makedirs(RAW_DIR, exist_ok=True)
    path = os.path.join(RAW_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(data)} records to {path}")


In [ ]:
print("Fetching posts (news articles)...")
posts = fetch_all("posts")
save_json(posts, "monze_posts.json")

In [ ]:
print("Fetching pages (static council info)...")
pages = fetch_all("pages")
save_json(pages, "monze_pages.json")

## 2. Data Cleaning and Preprocessing

The raw JSON from WordPress contains full HTML markup inside the `content`
and `title` fields (e.g. `<p>`, `<div>`, image galleries, inline styles).
This step strips that markup down to plain text using BeautifulSoup, and
also derives one additional feature: which of Monze District's three
constituencies (Monze Central, Moomba, Bweengwa) each article mentions,
detected via simple case-insensitive substring matching.

Empty static pages (template/utility pages with no real body content) are
dropped, since they add no informational value to the dataset.

The cleaned output is written as **pipe-separated (`|`) CSV files**, following
the naming convention required by the assignment:
`db-unza26-csc4792-[description].csv`.


In [ ]:
import csv
import re
from bs4 import BeautifulSoup

OUT_DIR = "datasets"
CONSTITUENCIES = ["Monze Central", "Moomba", "Bweengwa"]


def html_to_text(html: str) -> str:
    """Strip HTML tags/entities down to clean readable text."""
    soup = BeautifulSoup(html or "", "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()


def detect_constituencies(text: str) -> str:
    """Return semicolon-joined list of constituencies mentioned in the text."""
    found = [c for c in CONSTITUENCIES if c.lower() in text.lower()]
    return ";".join(found) if found else ""


def write_csv(rows, fieldnames, filename):
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, filename)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="|")
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {len(rows)} rows to {path}")

In [ ]:
# Clean and export the news articles
article_rows = []
for p in posts:
    content_text = html_to_text(p["content"]["rendered"])
    article_rows.append({
        "post_id": p["id"],
        "date": p["date"][:10],
        "title": html_to_text(p["title"]["rendered"]),
        "constituencies_mentioned": detect_constituencies(content_text),
        "content_text": content_text,
        "url": p["link"],
    })

write_csv(
    article_rows,
    ["post_id", "date", "title", "constituencies_mentioned", "content_text", "url"],
    "db-unza26-csc4792-monze_news_articles.csv",
)

In [ ]:
# Clean and export the static pages, dropping empty ones
page_rows = []
for p in pages:
    content_text = html_to_text(p["content"]["rendered"])
    if not content_text:
        continue
    page_rows.append({
        "page_id": p["id"],
        "title": html_to_text(p["title"]["rendered"]),
        "content_text": content_text,
        "url": p["link"],
    })

write_csv(
    page_rows,
    ["page_id", "title", "content_text", "url"],
    "db-unza26-csc4792-monze_static_pages.csv",
)

## 3. Locating Financial and Project Figures

The news articles contain the district's actual CDF (Constituency Development
Fund) activity — budget allocations, disbursement amounts, road-works
progress, and project statuses — but this information is embedded in free-form
prose rather than structured fields.

To surface it efficiently, every article was split into sentences and scanned
with a regular expression that matches:
- Kwacha amounts (e.g. `K3,278,013.69`, `K3.2 million`)
- Distances in kilometers (e.g. `15 km`)
- Percentages (e.g. `85%`)

This narrows 53 articles down to the ~20 that contain concrete figures,
without requiring a full manual re-read of every article.


In [ ]:
NUMBER_PATTERN = re.compile(
    r"(K\s?[\d,]+(\.\d+)?(\s?(million|billion))?"   # Kwacha amounts
    r"|\d+(\.\d+)?\s?(km|kilometers|kilometres)"      # distances
    r"|\d+(\.\d+)?\s?%"                                # percentages
    r")",
    re.IGNORECASE,
)


def split_sentences(text: str):
    return re.split(r"(?<=[.!?])\s+", text)


candidate_count = 0
for p in posts:
    text = html_to_text(p["content"]["rendered"])
    hits = [s for s in split_sentences(text) if NUMBER_PATTERN.search(s)]
    candidate_count += len(hits)

print(f"Total candidate sentences containing figures: {candidate_count}")

## 4. Curating the CDF Projects Dataset

The candidate sentences from Step 3 were manually reviewed (automated regex
extraction alone cannot reliably infer project name, constituency, sector,
and status from free-form news prose — this requires human judgement) and
organized into a structured table with consistent columns:

`project_name | constituency | sector | amount_kwacha | status | year | source_post_id | source_url`

Each row traces back to its source article via `source_post_id` and
`source_url`, so every figure in the dataset is independently verifiable
against the original council publication. Rows where the source sentence
gave a physical measure (e.g. kilometers completed) but no Kwacha figure are
recorded with `amount_kwacha = NA` rather than a fabricated value.

This is the richest of the three dataset files — structured, numeric, and
directly tied to the assignment's required data points (CDF allocations,
disbursements, project statuses, budgets, local revenue).


In [ ]:
import pandas as pd

cdf_df = pd.read_csv("datasets/db-unza26-csc4792-monze_cdf_projects.csv", sep="|")
print(f"CDF projects dataset: {len(cdf_df)} rows, {len(cdf_df.columns)} columns")
cdf_df.head(10)

## 5. Dataset Validation

A final check confirms all three output files meet the assignment's formatting
rules before upload to Kaggle:

- CSV format
- Pipe (`|`) delimiter
- Naming convention: `db-unza26-csc4792-[description].csv`


In [ ]:
dataset_files = [
    "db-unza26-csc4792-monze_news_articles.csv",
    "db-unza26-csc4792-monze_static_pages.csv",
    "db-unza26-csc4792-monze_cdf_projects.csv",
]

for fname in dataset_files:
    path = os.path.join("datasets", fname)
    df = pd.read_csv(path, sep="|")
    print(f"{fname}: {len(df)} rows, {len(df.columns)} columns — OK")

## Summary

| File | Rows | Description |
|---|---|---|
| `db-unza26-csc4792-monze_news_articles.csv` | 53 | All council news posts, HTML stripped, constituency-tagged |
| `db-unza26-csc4792-monze_static_pages.csv` | 34 | Static council pages (About, Contact, district info, etc.) |
| `db-unza26-csc4792-monze_cdf_projects.csv` | 46 | Curated CDF allocations, disbursements, and project records |

**Next steps:** upload these three files to Kaggle with documentation
referencing the exemplar dataset structure, push this notebook and its
source scripts to GitHub, and write the accompanying Data in Brief paper.
